## Configurações

In [4]:
# Extensão que verifica se um dos arquivos foi alterado
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from c3_1_instance_reweighing import instance_reweighing  # É preciso começar por ele para não dar problema com o torch vindo da aif360

# Bibliotecas
import pandas as pd
import joblib
import traceback
import os


# Variáveis auxiliares
from c0_1_configuracoes import(
  param_grid_perceptron_basico,
  param_grid_random_forest_basico,
  param_grid_regressao_logistica_basico,
  param_grid_xgboost_basico,
  param_grid_perceptron_completo,
  param_grid_random_forest_completo,
  param_grid_regressao_logistica_completo,
  param_grid_xgboost_completo,
  preprocessor_passthrough
)

from c0_2_cronometro import cronometro

# Funções auxiliares
from c1_6_enviesamento import enviesar
from c1_7_salvar_resultados import gerar_planilha, salvar_dicionario

# Algoritmos
from c2_1_random_forest import random_forest_GSCV
from c2_2_xgboost import xgboost_GSCV
from c2_3_regressao_logistica import regressao_logistica_GSCV
from c2_4_perceptron import perceptron_GSCV

# Técnicas de pré-processamento
from c3_1_instance_reweighing import instance_reweighing
from c3_2_disparate_impact_removal import disparate_impact_removal
from c3_3_synthetic_data_generation import synthetic_data_generation
from c3_4_suppression import suppression

# Técnicas de pós-processamento
from c4_1_threshold_optimization import threshold_optimization
from c4_2_calibration import calibration
from c4_3_reject_option_classification import reject_option_classification

# Interpretabilidade
from c5_0_interpretabilidade import gerar_interpretabilidade

caminho_resultado = './Resultados'

c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\inFairness\utils\ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\inFairness\utils\ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  monte_carlo_vect_ndcg = vmap(

## Cálculo

In [ ]:
# Chama a função que aplica todos os enviesamento considerando as variáveis sensiveis
# Não compensa salvar o arquivo pois seria muito pesado e leva apenas ~10 segundos para executar
datasets = enviesar()

In [ ]:
modo_completo = 0

param_grid_random_forest = param_grid_random_forest_completo if modo_completo else param_grid_random_forest_basico
param_grid_perceptron = param_grid_perceptron_completo if modo_completo else param_grid_perceptron_basico
param_grid_regressao_logistica = param_grid_regressao_logistica_completo if modo_completo else param_grid_regressao_logistica_basico
param_grid_xgboost = param_grid_xgboost_completo if modo_completo else param_grid_xgboost_basico

algoritmos = {
    'random_forest': {
        'nome_modelo': 'RANDOM FOREST',
        'funcao': random_forest_GSCV,
        'parametros': param_grid_random_forest
    },
    'perceptron': {
        'nome_modelo': 'PERCEPTRON',
        'funcao': perceptron_GSCV,
        'parametros': param_grid_perceptron
    },
    'regressão_logística': {
        'nome_modelo': 'REGRESSÃO LOGÍSTICA',
        'funcao': regressao_logistica_GSCV,
        'parametros': param_grid_regressao_logistica
    },
    'xgboost': {
        'nome_modelo': 'XGBOOST',
        'funcao': xgboost_GSCV,
        'parametros': param_grid_xgboost
    }
}

printar_tecnicas = False

# Parâmetros comuns para todos
parametros = {}
parametros['printar'] = True
parametros['cv_n_splits'] = 2

resultado_global = {}
sucesso = 0

# Percorre cada base de dados distinta
try:

  for df in datasets:

    print(f"\n\n=== INICIANDO ANÁLISE DO {df.upper()} ===\n")
    # Percorre cada tipo de enviesamento
    for tipo in datasets[df]:

      print(f"\n--- Analisando o tipo: {tipo} ---\n")

      banco = datasets[df][tipo]
      nome_banco = banco['nome_banco']

      # Separando dados de treino e teste
      X_train = banco['treino'].drop('target',axis=1)
      y_train = banco['treino']['target']

      X_test = banco['teste'].drop('target', axis=1)
      y_test = banco['teste']['target']

      # Passando os dados em dataframes já separados
      parametros_de_treino = parametros
      parametros_de_treino['X_train'] = X_train
      parametros_de_treino['X_test'] = X_test
      parametros_de_treino['y_train'] = y_train
      parametros_de_treino['y_test'] = y_test
      parametros_de_treino['preprocessor'] = preprocessor_passthrough # Já foram pré-processados antes
      parametros_de_treino['dados_sensiveis'] = banco['dados_sensiveis']

      resultado_dataset = {}
      resultado_dataset['smote'] = banco['smote']

      for nome_algoritmo in algoritmos:
        print(f"\n### Analisando o algoritmo: {nome_algoritmo.upper()} ###\n")

        algoritmo = algoritmos[nome_algoritmo]

        parametros_de_treino['param_grid'] = algoritmo['parametros']
        parametros_de_treino['nome_modelo'] = algoritmo['nome_modelo']
        parametros_de_treino['nome_base_de_dados'] = nome_banco + f" || {algoritmo['nome_modelo']}"

        desempenho = {}

        # Pré-processamento
        with cronometro() as timer:
          desempenho['instance_reweighing']               = instance_reweighing(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['instance_reweighing']['tempo'] = timer()

        with cronometro() as timer:
          desempenho['disparate_impact_removal']     = disparate_impact_removal(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['disparate_impact_removal']['tempo'] = timer()

        with cronometro() as timer:
          desempenho['synthetic_data_generation']   = synthetic_data_generation(algoritmo['funcao'], parametros_de_treino, colunas_discretas=banco['colunas_discretas'], printar=printar_tecnicas)
        desempenho['synthetic_data_generation']['tempo'] = timer()

        with cronometro() as timer:  
          desempenho['suppression']                               = suppression(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['suppression']['tempo'] = timer()

        # Pós-processamento
        with cronometro() as timer: 
          (_, desempenho['threshold_optimization'])                         = threshold_optimization(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['threshold_optimization']['tempo'] = timer()

        with cronometro() as timer: 
          (_, desempenho['calibration'])                                               = calibration(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['calibration']['tempo'] = timer()

        with cronometro() as timer: 
          (_, desempenho['reject_option_classification'])             = reject_option_classification(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['reject_option_classification']['tempo'] = timer()

        # Desempenho original (sem técnica)
        parametros_de_treino['nome_base_de_dados'] = parametros_de_treino['nome_base_de_dados'] + " || SEM TÉCNICA"
        with cronometro() as timer:
          (_, desempenho['sem_tecnica']) = algoritmo['funcao'](**parametros_de_treino)
        desempenho['sem_tecnica']['tempo'] = timer()

        resultado_dataset[f"{algoritmo['nome_modelo'].lower().replace(' ', '_')}"] = desempenho

      resultado_global[f"{nome_banco.lower().replace(' ', '_')}"] = resultado_dataset.copy()

except KeyboardInterrupt:
  traceback.print_exc()


except Exception:
  traceback.print_exc()

else:
  sucesso = 1

finally:
  print("\n\n\n-------------------------------------------")
  salvar_dicionario(resultado_global, caminho_resultado, sucesso)
  gerar_planilha(resultado_global, caminho_resultado)
  print("-------------------------------------------\n\n\n")



=== INICIANDO ANÁLISE DO DF1 ===


--- Analisando o tipo: original_sensitive_sexo ---


### Analisando o algoritmo: RANDOM_FOREST ###


----- DATASET 1 ORIGINAL COM SENSITIVE SEXO || RANDOM FOREST || INSTANCE REWEIGHING || RELATÓRIO DE CLASSIFICAÇÃO -----

              precision    recall  f1-score   support

           0       0.84      0.95      0.89      6805
           1       0.65      0.35      0.46      1958

    accuracy                           0.81      8763
   macro avg       0.74      0.65      0.67      8763
weighted avg       0.80      0.81      0.79      8763

ROC AUC: 0.7684
F1 Score: 0.4596
KS: 0.4099

----- DATASET 1 ORIGINAL COM SENSITIVE SEXO || RANDOM FOREST || INSTANCE REWEIGHING || MATRIZ DE CONFUSÃO -----

TN: 6437 | FP: 368
FN: 1264 | TP: 694

----- DATASET 1 ORIGINAL COM SENSITIVE SEXO || RANDOM FOREST || INSTANCE REWEIGHING || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.1129
Selection Rate (

c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\fairlearn\postprocessing\_interpolated_thresholder.py:154: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1. 1. 0. ... 0. 0. 1.]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  positive_probs[sensitive_feature_vector == a] = interpolated_predictions[



----- DATASET 1 ORIGINAL COM SENSITIVE SEXO || XGBOOST || THRESHOLD OPTIMIZATION || RELATÓRIO DE CLASSIFICAÇÃO -----

              precision    recall  f1-score   support

           0       0.84      0.94      0.89      6805
           1       0.65      0.36      0.46      1958

    accuracy                           0.81      8763
   macro avg       0.74      0.65      0.67      8763
weighted avg       0.79      0.81      0.79      8763

ROC AUC: 0.7692
F1 Score: 0.4624
KS: 0.4111

----- DATASET 1 ORIGINAL COM SENSITIVE SEXO || XGBOOST || THRESHOLD OPTIMIZATION || MATRIZ DE CONFUSÃO -----

TN: 6419 | FP: 386
FN: 1253 | TP: 705

----- DATASET 1 ORIGINAL COM SENSITIVE SEXO || XGBOOST || THRESHOLD OPTIMIZATION || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.1250
Selection Rate (Privilegiado):   0.1236
Demographic Parity Difference: 0.0014
Demographic Parity Ratio:    1.0113

----- True Positive Rate (Equal Opportunity) --

c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\aif360\algorithms\postprocessing\reject_option_classification.py:160: UserWarning: Unable to satisy fairness constraints
  warn("Unable to satisy fairness constraints")



----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || RANDOM FOREST || REJECT OPTION CLASSIFICATION || RELATÓRIO DE CLASSIFICAÇÃO -----

              precision    recall  f1-score   support

           0       0.84      0.93      0.88      6805
           1       0.62      0.40      0.48      1958

    accuracy                           0.81      8763
   macro avg       0.73      0.66      0.68      8763
weighted avg       0.79      0.81      0.79      8763

ROC AUC: 0.7544
F1 Score: 0.4832
KS: 0.3836

----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || RANDOM FOREST || REJECT OPTION CLASSIFICATION || MATRIZ DE CONFUSÃO -----

TN: 6324 | FP: 481
FN: 1181 | TP: 777

----- DATASET 1 SMOTE SIMPLES COM SENSITIVE SEXO || RANDOM FOREST || REJECT OPTION CLASSIFICATION || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.1062
Selection Rate (Privilegiado):   0.2022
Demographic Parity Difference: -0.0960
Demographic Parity Ratio:    0.5

...


----- DATASET 4 SMOTE SIMPLES COM SENSITIVE AGE || REGRESSÃO LOGÍSTICA || REJECT OPTION CLASSIFICATION || RELATÓRIO DE CLASSIFICAÇÃO -----

              precision    recall  f1-score   support

           0       0.82      0.82      0.82       200
           1       0.59      0.60      0.59        87

    accuracy                           0.75       287
   macro avg       0.71      0.71      0.71       287
weighted avg       0.75      0.75      0.75       287

ROC AUC: 0.7827
F1 Score: 0.5943
KS: 0.4466

----- DATASET 4 SMOTE SIMPLES COM SENSITIVE AGE || REGRESSÃO LOGÍSTICA || REJECT OPTION CLASSIFICATION || MATRIZ DE CONFUSÃO -----

TN: 164 | FP: 36
FN: 35 | TP: 52

----- DATASET 4 SMOTE SIMPLES COM SENSITIVE AGE || REGRESSÃO LOGÍSTICA || REJECT OPTION CLASSIFICATION || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.3033
Selection Rate (Privilegiado):   0.3091
Demographic Parity Difference: -0.0058
Demographic Parity Rat

c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\fairlearn\postprocessing\_interpolated_thresholder.py:154: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.    0.    1.    0.    0.    0.    0.    1.    1.    0.    0.    0.
 0.    0.    0.    0.412 0.    0.    1.    0.    0.    0.    0.    0.
 1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    1.    0.
 0.    0.    0.    1.    1.    0.    0.    1.    0.    0.    0.    0.
 0.    1.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.
 1.    0.    0.    0.    0.    1.    0.    0.    0.    1.    0.    0.
 0.    0.    1.    0.    0.    0.    0.    1.    0.    1.    1.    0.
 0.    0.    1.    0.    0.    0.    0.    1.    0.    0.    0.    0.
 1.    0.    0.    1.    0.    1.    0.    1.    0.    1.    0.    1.
 0.412 0.    0.    0.    0.    0.412 1.    0.    1.    1.    1.    0.
 1.    0.   ]' has dtype incompatible with float32, please 


----- DATASET 4 SMOTE SIMPLES COM SENSITIVE AGE || XGBOOST || THRESHOLD OPTIMIZATION || RELATÓRIO DE CLASSIFICAÇÃO -----

              precision    recall  f1-score   support

           0       0.77      0.90      0.83       200
           1       0.63      0.39      0.48        87

    accuracy                           0.75       287
   macro avg       0.70      0.65      0.66       287
weighted avg       0.73      0.75      0.73       287

ROC AUC: 0.7618
F1 Score: 0.4823
KS: 0.4576

----- DATASET 4 SMOTE SIMPLES COM SENSITIVE AGE || XGBOOST || THRESHOLD OPTIMIZATION || MATRIZ DE CONFUSÃO -----

TN: 180 | FP: 20
FN: 53 | TP: 34

----- DATASET 4 SMOTE SIMPLES COM SENSITIVE AGE || XGBOOST || THRESHOLD OPTIMIZATION || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.2787
Selection Rate (Privilegiado):   0.1212
Demographic Parity Difference: 0.1575
Demographic Parity Ratio:    2.2992

----- True Positive Rate (Equal Opportun

c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\aif360\algorithms\postprocessing\reject_option_classification.py:160: UserWarning: Unable to satisy fairness constraints
  warn("Unable to satisy fairness constraints")



----- DATASET 4 SMOTE SIMPLES COM SENSITIVE AGE || XGBOOST || REJECT OPTION CLASSIFICATION || RELATÓRIO DE CLASSIFICAÇÃO -----

              precision    recall  f1-score   support

           0       0.79      0.81      0.80       200
           1       0.54      0.51      0.52        87

    accuracy                           0.72       287
   macro avg       0.66      0.66      0.66       287
weighted avg       0.71      0.72      0.72       287

ROC AUC: 0.7618
F1 Score: 0.5207
KS: 0.4576

----- DATASET 4 SMOTE SIMPLES COM SENSITIVE AGE || XGBOOST || REJECT OPTION CLASSIFICATION || MATRIZ DE CONFUSÃO -----

TN: 162 | FP: 38
FN: 43 | TP: 44

----- DATASET 4 SMOTE SIMPLES COM SENSITIVE AGE || XGBOOST || REJECT OPTION CLASSIFICATION || JUSTIÇA DO MODELO -----

----- Demographic Parity (Selection Rate) -----
Selection Rate (Desprivilegiado): 0.2787
Selection Rate (Privilegiado):   0.2909
Demographic Parity Difference: -0.0122
Demographic Parity Ratio:    0.9580

----- True Positive R

## Controle manual

In [6]:
dicionario = './Resultados/resultado_global_dict_03_01_2026_19_50.joblib'

if os.path.isfile(dicionario):
  rf = joblib.load(dicionario)
  print("Dicionário recuperado")

Dicionário recuperado


In [10]:
gerar_interpretabilidade(rf, '.')

Iniciando cálculo de interpretabilidade...
Processando interpretabilidade para: DATASET 1 ORIGINAL COM SENSITIVE SEXO || RANDOM FOREST || INSTANCE REWEIGHING
  > SHAP concluído em 119.77s
  > Permutation Importance concluída em 118.85s
  > LIME concluído em 141.99s
Processando interpretabilidade para: DATASET 1 ORIGINAL COM SENSITIVE SEXO || RANDOM FOREST || DISPARATE IMPACT REMOVAL
  > SHAP concluído em 117.11s
  > Permutation Importance concluída em 91.41s
  > LIME concluído em 115.95s
Processando interpretabilidade para: DATASET 1 ORIGINAL COM SENSITIVE SEXO || RANDOM FOREST || DADOS DE TESTE COM SDG || SYNTHETIC DATA GENERATION
  > SHAP concluído em 112.08s
  > Permutation Importance concluída em 104.45s
  > LIME concluído em 97.85s
Processando interpretabilidade para: DATASET 1 ORIGINAL COM SENSITIVE SEXO || RANDOM FOREST || SUPPRESSION
  > SHAP concluído em 105.89s
  > Permutation Importance concluída em 99.67s


KeyboardInterrupt: 

In [37]:
gerar_planilha(rf, caminho_resultado)

Arquivo Excel com 0 linhas de dados salvo em ./Resultados/resultado_global_12_12_2025_16_31.xlsx


## Visualizar o dicionário de resultado

In [11]:
def listar_hierarquia_ordenada(dicionario, nivel=0):
    # Ordenar as chaves
    for chave, valor in sorted(dicionario.items()):
        indentacao = "    " * nivel
        print(f"{indentacao}{chave}")
        
        # Acessa os demais dicionários recursivamente
        if isinstance(valor, dict):
            listar_hierarquia_ordenada(valor, nivel + 1)